# Train a detector on your Workshop data

This notebook trains a YOLO detector on images you reviewed in openskycam and publishes it to your model registry, where workflows can use it.

1. **Freeze a dataset version** of each bucket you want to train on: open the bucket in Workshop and choose **Freeze version**, or call the API ([Buckets and datasets](https://docs.openskycam.org/workshop/datasets/)). Turn on rejected images if you want them to teach the model what *not* to flag.
2. **Create two keys** under **API access** on your website: one that can read footage for the cameras in those versions (Agent or Footage), and one with **Publish trained models**.
3. **Run it on a GPU.** In Colab, choose *Runtime → Change runtime type → GPU*. Training at 1920 px needs a large GPU; 1280 or 640 runs on smaller ones.

To try a published model before training your own, see the [night detector](https://github.com/Deibler/openskycam/tree/main/models/night-detector).

In [ ]:
# Your deployment
API = "https://api.example.com"     # your API_HOST
VERSIONS = ["planes-2026-10-25"]    # dataset version IDs to train on; their class names are merged

# The model
BASE_MODEL = "yolo26s.pt"           # Ultralytics weights to start from, or a path to your own .pt
IMGSZ = 1280                        # input size: a multiple of 32, at most 2048. Workflows run the model at this size.
EPOCHS = 50
REJECTED_AS_EMPTY = True            # train rejected images as empty sky, so the model learns what not to flag

# Publishing
FAMILY = "my-detector"              # lowercase letters, digits, dots, underscores or hyphens
VERSION = "v1"
NAME = "My detector"
PUBLISH = True

LIMIT = None                        # a number of images per split, for a quick end-to-end test; never publishes

## Setup

Keys come from Colab Secrets or environment variables named `OPENSKYCAM_READ_KEY` and `OPENSKYCAM_PUBLISH_KEY`, so they never appear in the notebook. Ultralytics is pinned to the version the processor runs, so the processor can load what you train.

In [ ]:
import hashlib, importlib.util, json, os, shutil, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

ULTRALYTICS = "8.4.58"  # the processor's version (processor/requirements.txt)
if importlib.util.find_spec("ultralytics") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}"], check=True)
if importlib.util.find_spec("skyvision_processor") is None:  # only for publishing; it needs no other packages
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "git+https://github.com/Deibler/openskycam#subdirectory=processor"], check=True)
import cv2, numpy as np, torch, ultralytics
from ultralytics import YOLO

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def secret(name):
    value = ""
    if userdata:
        try:
            value = userdata.get(name) or ""
        except Exception:  # not set, or this notebook has no access to it
            pass
    return (value or os.environ.get(name, "")).strip()


READ_KEY, PUBLISH_KEY = secret("OPENSKYCAM_READ_KEY"), secret("OPENSKYCAM_PUBLISH_KEY")
assert READ_KEY, "Set OPENSKYCAM_READ_KEY: a key that can read the cameras in your dataset versions"
assert not PUBLISH or LIMIT or PUBLISH_KEY, "Set OPENSKYCAM_PUBLISH_KEY (Publish trained models), or PUBLISH = False"
assert IMGSZ % 32 == 0 and 32 <= IMGSZ <= 2048, "IMGSZ must be a multiple of 32, at most 2048"
if ultralytics.__version__ != ULTRALYTICS:
    print(f"Ultralytics {ultralytics.__version__} is installed; the processor runs {ULTRALYTICS}. Test the model before relying on it.")
DEVICE = 0 if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
# Absolute: Ultralytics puts a relative project folder under its own runs directory instead.
WORK = (Path("openskycam-training") / f"{FAMILY}-{VERSION}").resolve()
WORK.mkdir(parents=True, exist_ok=True)
print("training on", DEVICE, "in", WORK)


def get(path, attempts=4):
    for attempt in range(attempts):
        try:
            request = urllib.request.Request(API + path, headers={"Authorization": f"Bearer {READ_KEY}",
                                                                  "User-Agent": "openskycam-training-notebook"})
            with urllib.request.urlopen(request, timeout=120) as response:
                return response.read()
        except urllib.error.HTTPError as error:
            if error.code < 500 or attempt == attempts - 1:
                raise RuntimeError(f"GET {path}: HTTP {error.code} {error.read()[:200]!r}") from None
        except OSError:
            if attempt == attempts - 1:
                raise
        time.sleep(2 ** attempt)

## Download the dataset

Each version's manifest lists its images, boxes, reviews and a split. A split covers whole camera-days, so near-identical frames of one night never land on both sides. Images are checked against their SHA-256 and shrunk to `IMGSZ` on the long side with the same resize the processor uses, so the model trains on what it will see.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

manifests = [json.loads(get(f"/v1/workshop/versions/{v}"))["manifest"] for v in VERSIONS]
CLASSES = sorted({name for m in manifests for name in m["class_names"]})
DATA = WORK / f"data-{IMGSZ}"
items = []
for manifest in manifests:
    for item in manifest["items"]:
        if item["review"] == "rejected" and not REJECTED_AS_EMPTY:
            continue
        boxes = [] if item["review"] == "rejected" else [
            (CLASSES.index(a["class_name"]), a["box"]) for a in item["annotations"] if "box" in a]
        items.append({**item, "boxes": boxes})
if LIMIT:
    items = [i for split in ("train", "validation", "test") for i in [i for i in items if i["split"] == split][:LIMIT]]


def shrink(image):
    # The processor's own resize: INTER_AREA down to IMGSZ on the long side, never up.
    scale = IMGSZ / max(image.shape[:2])
    if scale >= 1:
        return image
    return cv2.resize(image, (max(1, int(image.shape[1] * scale)), max(1, int(image.shape[0] * scale))),
                      interpolation=cv2.INTER_AREA)


def prepare(item):
    folder = "val" if item["split"] == "validation" else item["split"]
    image = DATA / "images" / folder / f"{item['sha256']}.jpg"
    if not image.exists():
        data = get(item["media_path"])
        assert hashlib.sha256(data).hexdigest() == item["sha256"], f"{item['item_id']}: checksum mismatch"
        frame = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        part = image.with_suffix(".part.jpg")
        cv2.imwrite(str(part), shrink(frame), [cv2.IMWRITE_JPEG_QUALITY, 95])
        part.replace(image)
    lines = []
    for k, (x1, y1, x2, y2) in item["boxes"]:  # YOLO: class, centre x, centre y, width, height, as fractions
        x1, y1, x2, y2 = (min(max(v, 0.0), 1.0) for v in (x1, y1, x2, y2))
        lines.append(f"{k} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} {x2 - x1:.6f} {y2 - y1:.6f}")
    (DATA / "labels" / folder / f"{item['sha256']}.txt").write_text("\n".join(lines))


for kind in ("images", "labels"):
    for folder in ("train", "val", "test"):
        (DATA / kind / folder).mkdir(parents=True, exist_ok=True)
with ThreadPoolExecutor(8) as pool:
    list(pool.map(prepare, items))
(DATA / "data.yaml").write_text(json.dumps({"path": str(DATA.resolve()), "train": "images/train", "val": "images/val",
                                            "test": "images/test", "names": dict(enumerate(CLASSES))}))
counts = {s: sum(i["split"] == s for i in items) for s in ("train", "validation", "test")}
print("classes:", CLASSES)
print("images:", counts)
assert counts["train"] and counts["validation"], "Training needs train and validation images; freeze a version with more days"

## Train

Training resumes from the last finished epoch if the notebook was interrupted.

In [ ]:
RUN = WORK / "train"
last, best = RUN / "weights" / "last.pt", RUN / "weights" / "best.pt"
if last.exists() and not best.exists():
    YOLO(str(last)).train(resume=True)
elif not best.exists():
    YOLO(BASE_MODEL).train(data=str(DATA / "data.yaml"), imgsz=IMGSZ, epochs=1 if LIMIT else EPOCHS, patience=15,
                           batch=-1 if DEVICE == 0 else 2, device=DEVICE, seed=0, fliplr=0.5, flipud=0.0,
                           project=str(WORK), name="train", exist_ok=True, plots=not LIMIT, workers=0 if LIMIT else 8)
assert best.exists(), "training did not produce best.pt"

## Score on the test split

The test images were never used for training or for choosing the best epoch, so these numbers are the honest ones to compare models by.

In [ ]:
model = YOLO(str(best))
evaluations = []
if counts["test"]:
    metrics = model.val(data=str(DATA / "data.yaml"), split="test", imgsz=IMGSZ, device=DEVICE, plots=False).results_dict
    evaluations.append({"label": "Test split", "split": "test", "input_size": IMGSZ,
                        "dataset": ", ".join(VERSIONS) + f": {counts['test']} images",
                        "metrics": {k: round(float(v), 5) for k, v in metrics.items()}, "independently_reproduced": False})
    print(json.dumps(evaluations[-1]["metrics"], indent=1))
else:
    print("This dataset has no test images; freeze a version that covers more days to get a test score.")

## Publish

Publishing uploads the checkpoint in checksum-verified chunks and creates an immutable model version. Afterwards, choose `FAMILY/VERSION` in a workflow's model step and start with a test run.

In [ ]:
from skyvision_processor.service.modelInferenceService import publish
from skyvision_processor.utils.httpSharedUtil import Client

card = {
    "family": FAMILY, "version": VERSION, "name": NAME,
    "description": f"{BASE_MODEL} fine-tuned at {IMGSZ} px on dataset versions {', '.join(VERSIONS)}.",
    "task": "object-detection", "adapter": "ultralytics",
    "classes": [{"id": int(k), "name": v} for k, v in sorted(model.names.items())],
    "input": {"kind": "image", "channels": 3, "color_space": "RGB", "training_size": IMGSZ,
              "preprocessing": "cv2.INTER_AREA to the long side, as the processor does, then Ultralytics letterbox"},
    "evaluations": evaluations,
    "caveats": [f"Trained with Ultralytics {ultralytics.__version__}.", "Metrics come from this notebook, not a Workshop run."],
    "dependencies": [],
}
(WORK / "model.json").write_text(json.dumps(card, indent=2))
if LIMIT or not PUBLISH:
    print("Not published:", "a LIMIT test run" if LIMIT else "PUBLISH = False", f"\nWeights: {best}\nCard: {WORK / 'model.json'}")
else:
    published = publish(Client(API, PUBLISH_KEY), card, [best])
    print("Published", f"{published['family']}/{published['version']}")